# GlowWise Skincare - EDA atualizada

Este notebook documenta a analise exploratoria dos dados usados no Desafio 02. A EDA atual esta alinhada aos arquivos gerados pela pipeline em `data/processed` e aos artefatos de apresentacao em `reports/eda`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parent

RAW_PATH = PROJECT_DIR / "data" / "raw" / "produtos_coletados_historico.csv"
SANITIZED_PATH = PROJECT_DIR / "data" / "processed" / "produtos_sanitizados.csv"
ML_PATH = PROJECT_DIR / "data" / "processed" / "produtos_ml.csv"
REPORTS_EDA = PROJECT_DIR / "reports" / "eda"


In [ ]:
df_raw = pd.read_csv(RAW_PATH)
df = pd.read_csv(SANITIZED_PATH)
df_ml = pd.read_csv(ML_PATH)

print(f"Bruto consolidado: {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas")
print(f"Sanitizado: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"Dataset ML: {df_ml.shape[0]} linhas x {df_ml.shape[1]} colunas")


## Origem dos dados

O fluxo atual da base e: `data/raw/execucoes` -> `produtos_coletados_historico.csv` -> `produtos_sanitizados.csv` -> `produtos_ml.csv`. O arquivo `produtos_ml.csv` e o dataset usado no treino, avaliacao, curvas ROC/PR e relatorios dos slides.


In [ ]:
df_ml.head()


## Estrutura do dataset ML

O problema foi modelado como classificacao binaria: `0 = nao_vale_comprar` e `1 = vale_comprar`. As features oficiais sao `texto_produto`, `categoria_busca`, `loja` e `preco`.


In [ ]:
features = ["texto_produto", "categoria_busca", "loja", "preco"]
target = "vale_comprar"

print("Features:", features)
print("Target:", target)
print("Valores ausentes no dataset ML:", int(df_ml.isna().sum().sum()))
df_ml[features + [target]].head()


## Classe alvo e balanceamento

A base atual tem 113 instancias. A classe positiva `vale_comprar` tem 64 registros e a classe negativa tem 49 registros. Isso indica uma base levemente inclinada para a classe positiva, mas sem desbalanceamento extremo.


In [ ]:
alvo = df_ml[target].map({0: "nao_vale_comprar", 1: "vale_comprar"})
pd.DataFrame({
    "registros": alvo.value_counts(),
    "percentual": (alvo.value_counts(normalize=True) * 100).round(1),
})


In [ ]:
ax = alvo.value_counts().plot(kind="bar", figsize=(6, 4), title="Distribuicao da classe alvo")
ax.set_xlabel("Classe")
ax.set_ylabel("Quantidade")
plt.xticks(rotation=0)
plt.show()


## Distribuicao por categoria e loja

A EDA mostra concentracao em hidratacao, limpeza e protecao solar. A categoria `tratamento` ainda tem poucas amostras e deve ser expandida em novas coletas.


In [ ]:
df_ml.groupby("categoria_busca")["preco"].agg(["count", "min", "median", "mean", "max"]).round(2)


In [ ]:
df_ml["loja"].value_counts()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df_ml["categoria_busca"].value_counts().plot(kind="bar", ax=axes[0], title="Produtos por categoria")
df_ml["loja"].value_counts().plot(kind="bar", ax=axes[1], title="Produtos por loja")
for ax in axes:
    ax.set_ylabel("Quantidade")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


## Analise de precos

O preco e a feature numerica do pipeline. A distribuicao por categoria ajuda a entender a heuristica inicial do rotulo e tambem possiveis outliers de produtos premium.


In [ ]:
df_ml["preco"].describe().round(2)


In [ ]:
df_ml.boxplot(column="preco", by="categoria_busca", figsize=(9, 5), grid=False)
plt.title("Precos por categoria")
plt.suptitle("")
plt.xlabel("Categoria")
plt.ylabel("Preco")
plt.xticks(rotation=20)
plt.show()


## Qualidade dos dados

A limpeza tratou precos, textos, duplicidades e links de redirecionamento. O dataset ML final nao possui valores ausentes.


In [ ]:
checks = {
    "valores_ausentes_ml": int(df_ml.isna().sum().sum()),
    "links_criteo_no_bruto": int(df_raw["link"].str.contains("criteo.com", case=False, na=False).sum()) if "link" in df_raw else 0,
    "links_criteo_apos_limpeza": int(df["link"].str.contains("criteo.com", case=False, na=False).sum()),
    "links_canonicos_duplicados": int(df["link_canonico"].duplicated().sum()),
    "registros_sem_preco": int(df["preco"].isna().sum()),
}
checks


## Artefatos atualizados da EDA

Os graficos e o resumo textual da EDA agora sao gerados por `ml/gerar_relatorio_eda.py`, com saida em `reports/eda`. Esses artefatos devem ser usados nos slides quando precisar demonstrar dataset, atributos, alvo, instancias e balanceamento.


In [ ]:
print("Resumo EDA:", REPORTS_EDA / "resumo_eda.md")
print("Graficos:")
for path in sorted(REPORTS_EDA.glob("*.png")):
    print("-", path.name)


## Conclusao da EDA

O dataset atual tem 113 instancias, 64 positivas e 49 negativas. A base esta pronta para o pipeline de classificacao binaria, mas ainda e pequena. A principal melhoria para proximas versoes e ampliar novas coletas, principalmente na categoria `tratamento`, e substituir gradualmente a heuristica inicial por rotulos revisados com feedback humano.
